# Task 2.2 — Khảo sát kiểu dữ liệu, ý nghĩa biến, tỷ lệ y=yes, unknown và pdays=-1

**DBM301 · Assignment 1 · Đề 04 · Group 3**

Mục tiêu:
- Hiểu kiểu dữ liệu và ý nghĩa từng biến trong `bank-full.csv`
- Tính tỷ lệ `y=yes` (biến mục tiêu)
- Xác định phân phối giá trị `unknown` theo từng biến
- Xác định ý nghĩa đặc biệt của `pdays=-1`
- Lập **bảng quyết định xử lý** cho từng biến (đầu ra của task 2.3)

## 0. Import thư viện

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for script execution
import matplotlib.pyplot as plt
import seaborn as sns

from dbm301.config import RAW_DATA_DIR, FIGURES_DIR, PROCESSED_DATA_DIR

# Tạo thư mục output nếu chưa có
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Thu vien da san sang.')
print(f'Figures dir: {FIGURES_DIR}')

2026-10-09 17:14:06.334 | INFO     | dbm301.config:<module>:11 - PROJ_ROOT path is: C:\Users\Admin\Documents\DBM301\Assignment1_de4\dbm301_asm_group3_de4


Thu vien da san sang.
Figures dir: C:\Users\Admin\Documents\DBM301\Assignment1_de4\dbm301_asm_group3_de4\reports\figures


## 1. Đọc dữ liệu

In [2]:
df = pd.read_csv(RAW_DATA_DIR / 'bank-full.csv', sep=';')

print(f'So quan sat : {df.shape[0]:,}')
print(f'So thuoc tinh: {df.shape[1]}')
df.head()

So quan sat : 45,211
So thuoc tinh: 17


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


## 2. Kiểu dữ liệu từng cột

In [3]:
dtype_summary = pd.DataFrame({
    'Kieu pandas'   : df.dtypes,
    'Phan loai'     : df.dtypes.apply(lambda t: 'So (lien tuc)' if t in ['int64','float64'] else 'Phan loai (nominal)'),
    'So gia tri duy nhat': df.nunique(),
    'Vi du'         : df.apply(lambda c: list(c.unique()[:3]))
})
print(dtype_summary.to_string())

          Kieu pandas            Phan loai  So gia tri duy nhat                                   Vi du
age             int64        So (lien tuc)                   77                            [58, 44, 33]
job            object  Phan loai (nominal)                   12  [management, technician, entrepreneur]
marital        object  Phan loai (nominal)                    3             [married, single, divorced]
education      object  Phan loai (nominal)                    4          [tertiary, secondary, unknown]
default        object  Phan loai (nominal)                    2                               [no, yes]
balance         int64        So (lien tuc)                 7168                           [2143, 29, 2]
housing        object  Phan loai (nominal)                    2                               [yes, no]
loan           object  Phan loai (nominal)                    2                               [no, yes]
contact        object  Phan loai (nominal)                    3 

## 3. Từ điển dữ liệu (Data Dictionary)

Theo tài liệu UCI Bank Marketing:

In [4]:
data_dict = {
    'age'      : ('So - Lien tuc',            'Tuoi cua khach hang'),
    'job'      : ('Phan loai - Nominal',       'Loai cong viec (admin., technician, ...)'),
    'marital'  : ('Phan loai - Nominal',       'Tinh trang hon nhan (married, single, divorced)'),
    'education': ('Phan loai - Ordinal',       'Trinh do hoc van (primary, secondary, tertiary)'),
    'default'  : ('Phan loai - Binary',        'Co no xau khong? (yes/no)'),
    'balance'  : ('So - Lien tuc',             'So du tai khoan tiet kiem trung binh hang nam (Euro)'),
    'housing'  : ('Phan loai - Binary',        'Co vay mua nha khong? (yes/no)'),
    'loan'     : ('Phan loai - Binary',        'Co vay ca nhan khong? (yes/no)'),
    'contact'  : ('Phan loai - Nominal',       'Phuong thuc lien lac (telephone, cellular, unknown)'),
    'day'      : ('So - Roi rac',              'Ngay lien he cuoi trong thang (1-31)'),
    'month'    : ('Phan loai - Ordinal',       'Thang lien he cuoi (jan-dec)'),
    'duration' : ('So - Lien tuc',             '** LOAI KHOI TAP ITEM ** Thoi gian cuoc goi (giay) - chi biet sau khi goi xong'),
    'campaign' : ('So - Roi rac',              'So lan lien he trong chien dich nay'),
    'pdays'    : ('So - Roi rac',              'So ngay tu lan lien he truoc. ** -1 = chua tung lien he truoc **'),
    'previous' : ('So - Roi rac',              'So lan lien he truoc chien dich nay'),
    'poutcome' : ('Phan loai - Nominal',       'Ket qua chien dich truoc (success, failure, other, unknown)'),
    'y'        : ('Phan loai - Binary (MUC TIEU)', 'Khach hang co dang ky tien gui co ky han? (yes/no)'),
}

dict_df = pd.DataFrame.from_dict(
    data_dict, orient='index', columns=['Kieu / Thang do', 'Y nghia']
)
dict_df.index.name = 'Bien'
print(dict_df.to_string())

                         Kieu / Thang do                                                                         Y nghia
Bien                                                                                                                    
age                        So - Lien tuc                                                             Tuoi cua khach hang
job                  Phan loai - Nominal                                        Loai cong viec (admin., technician, ...)
marital              Phan loai - Nominal                                 Tinh trang hon nhan (married, single, divorced)
education            Phan loai - Ordinal                                 Trinh do hoc van (primary, secondary, tertiary)
default               Phan loai - Binary                                                       Co no xau khong? (yes/no)
balance                    So - Lien tuc                            So du tai khoan tiet kiem trung binh hang nam (Euro)
housing               Phan loai 

## 4. Tỷ lệ y=yes (biến mục tiêu)

In [5]:
y_counts = df['y'].value_counts()
y_pct    = df['y'].value_counts(normalize=True) * 100

y_summary = pd.DataFrame({'So luong': y_counts, 'Ty le (%)': y_pct})
print('=== Phan phoi bien muc tieu y ===')
print(y_summary.to_string())

# Biểu đồ
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

y_counts.plot(kind='bar', ax=axes[0], color=['#d9534f', '#5cb85c'], edgecolor='black')
axes[0].set_title('So luong quan sat theo y')
axes[0].set_xlabel('y'); axes[0].set_ylabel('So quan sat')
axes[0].tick_params(axis='x', rotation=0)
for bar, val in zip(axes[0].patches, y_counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{val:,}', ha='center', fontsize=11)

axes[1].pie(y_counts, labels=y_counts.index, autopct='%1.1f%%',
            colors=['#d9534f', '#5cb85c'], startangle=90)
axes[1].set_title('Ty le y=yes / y=no')

plt.suptitle('Bien muc tieu y - Dang ky tien gui co ky han', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '01_y_distribution.png', dpi=150, bbox_inches='tight')
plt.close()

print(f"\n** Du lieu MAT CAN BANG: y=yes chi chiem {y_pct['yes']:.1f}% **")
print(f"Da luu: {FIGURES_DIR / '01_y_distribution.png'}")

=== Phan phoi bien muc tieu y ===
     So luong  Ty le (%)
y                       
no      39922    88.3015
yes      5289    11.6985

** Du lieu MAT CAN BANG: y=yes chi chiem 11.7% **
Da luu: C:\Users\Admin\Documents\DBM301\Assignment1_de4\dbm301_asm_group3_de4\reports\figures\01_y_distribution.png


## 5. Phân phối giá trị 'unknown' theo từng biến

In [6]:
cat_cols = df.select_dtypes(include='object').columns.tolist()

unknown_summary = []
for col in cat_cols:
    n_unknown = (df[col] == 'unknown').sum()
    pct = n_unknown / len(df) * 100
    unknown_summary.append({'Bien': col, 'So unknown': n_unknown, 'Ty le (%)': round(pct, 2)})

unknown_df = pd.DataFrame(unknown_summary).set_index('Bien')
unknown_df = unknown_df[unknown_df['So unknown'] > 0].sort_values('So unknown', ascending=False)

print('=== Gia tri unknown theo tung bien ===')
print(unknown_df.to_string())

# Biểu đồ
if not unknown_df.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    unknown_df['Ty le (%)'].plot(kind='barh', ax=ax, color='#f0ad4e', edgecolor='black')
    ax.set_xlabel('Ty le unknown (%)')
    ax.set_title('Ty le gia tri "unknown" theo tung bien phan loai')
    for i, v in enumerate(unknown_df['Ty le (%)']):
        ax.text(v + 0.2, i, f'{v:.1f}%', va='center')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '02_unknown_distribution.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Da luu: {FIGURES_DIR / '02_unknown_distribution.png'}")

print('\n** unknown KHONG phai gia tri thieu binh thuong - la mot gia tri thuc su co y nghia rieng. **')

=== Gia tri unknown theo tung bien ===
           So unknown  Ty le (%)
Bien                            
poutcome        36959    81.7500
contact         13020    28.8000
education        1857     4.1100
job               288     0.6400
Da luu: C:\Users\Admin\Documents\DBM301\Assignment1_de4\dbm301_asm_group3_de4\reports\figures\02_unknown_distribution.png

** unknown KHONG phai gia tri thieu binh thuong - la mot gia tri thuc su co y nghia rieng. **


## 6. Ý nghĩa đặc biệt của pdays = -1

In [7]:
n_minus1   = (df['pdays'] == -1).sum()
pct_minus1 = n_minus1 / len(df) * 100

print(f'pdays = -1 (chua tung lien he truoc): {n_minus1:,} quan sat ({pct_minus1:.1f}%)')
print(f'pdays != -1 (da tung lien he truoc) : {len(df) - n_minus1:,} quan sat ({100-pct_minus1:.1f}%)')
print()

# Phân phối pdays khi != -1
pdays_valid = df[df['pdays'] != -1]['pdays']
print('Thong ke pdays (khi != -1):')
print(pdays_valid.describe())

# So sánh tỷ lệ y=yes
print('\n=== Ty le y=yes theo nhom pdays ===')
df['pdays_group'] = df['pdays'].apply(lambda x: 'Chua lien he truoc (pdays=-1)' if x == -1 else 'Da lien he truoc')
yes_rate_by_pdays = df.groupby('pdays_group')['y'].apply(lambda x: (x=='yes').mean() * 100).round(2)
print(yes_rate_by_pdays.to_string())
df.drop(columns='pdays_group', inplace=True)

print('\n** pdays=-1 KHONG phai so do. Can tao bien nhi phan: contacted_before (yes/no). **')

pdays = -1 (chua tung lien he truoc): 36,954 quan sat (81.7%)
pdays != -1 (da tung lien he truoc) : 8,257 quan sat (18.3%)

Thong ke pdays (khi != -1):
count   8257.0000
mean     224.5777
std      115.3440
min        1.0000
25%      133.0000
50%      194.0000
75%      327.0000
max      871.0000
Name: pdays, dtype: float64

=== Ty le y=yes theo nhom pdays ===
pdays_group
Chua lien he truoc (pdays=-1)    9.1600
Da lien he truoc                23.0700

** pdays=-1 KHONG phai so do. Can tao bien nhi phan: contacted_before (yes/no). **


## 7. Thống kê mô tả các biến số

In [8]:
num_cols = ['age', 'balance', 'duration', 'campaign', 'pdays', 'previous']
print('=== Thong ke mo ta bien so ===')
print(df[num_cols].describe().T.to_string())

=== Thong ke mo ta bien so ===
              count      mean       std        min      25%      50%       75%         max
age      45211.0000   40.9362   10.6188    18.0000  33.0000  39.0000   48.0000     95.0000
balance  45211.0000 1362.2721 3044.7658 -8019.0000  72.0000 448.0000 1428.0000 102127.0000
duration 45211.0000  258.1631  257.5278     0.0000 103.0000 180.0000  319.0000   4918.0000
campaign 45211.0000    2.7638    3.0980     1.0000   1.0000   2.0000    3.0000     63.0000
pdays    45211.0000   40.1978  100.1287    -1.0000  -1.0000  -1.0000   -1.0000    871.0000
previous 45211.0000    0.5803    2.3034     0.0000   0.0000   0.0000    0.0000    275.0000


## 8. Tỷ lệ y=yes theo từng giá trị biến phân loại

In [9]:
analyze_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
overall_yes_rate = df['y'].eq('yes').mean() * 100

n_cols = 3
n_rows = -(-len(analyze_cols) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(analyze_cols):
    yes_rate = df.groupby(col)['y'].apply(lambda x: (x=='yes').mean() * 100).sort_values(ascending=False)
    yes_rate.plot(kind='bar', ax=axes[i], color='#5bc0de', edgecolor='black')
    axes[i].set_title(f'Ty le y=yes theo {col}')
    axes[i].set_ylabel('Ty le y=yes (%)')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].axhline(y=overall_yes_rate, color='red', linestyle='--', alpha=0.7, label=f'TB chung {overall_yes_rate:.1f}%')
    axes[i].legend(fontsize=8)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Ty le y=yes theo tung bien phan loai', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '03_yes_rate_by_category.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"Da luu: {FIGURES_DIR / '03_yes_rate_by_category.png'}")

Da luu: C:\Users\Admin\Documents\DBM301\Assignment1_de4\dbm301_asm_group3_de4\reports\figures\03_yes_rate_by_category.png


## 9. Bảng quyết định xử lý từng biến (đầu ra task 2.3)

In [10]:
processing_plan = [
    ('age',       'Lien tuc',                     'Discretize thanh nhom (young/middle/senior)',    'Can de tao item dang age=young cho ARM'),
    ('job',       'Co gia tri unknown',            'Giu unknown la mot gia tri rieng biet',         'unknown co y nghia rieng, khong impute'),
    ('marital',   'Nominal, khong co unknown',     'Giu nguyen, encode attribute=value',            'Da la gia tri roi rac'),
    ('education', 'Co gia tri unknown, ordinal',   'Giu unknown la mot gia tri rieng biet',         'unknown co y nghia rieng, khong impute'),
    ('default',   'Binary yes/no',                 'Giu nguyen, encode attribute=value',            'Da la gia tri roi rac'),
    ('balance',   'Lien tuc, co so am',            'Discretize thanh nhom (negative/low/mid/high)', 'Can roi rac hoa de tao item'),
    ('housing',   'Binary yes/no',                 'Giu nguyen',                                    'Da la gia tri roi rac'),
    ('loan',      'Binary yes/no',                 'Giu nguyen',                                    'Da la gia tri roi rac'),
    ('contact',   'Co gia tri unknown',            'Giu unknown la mot gia tri rieng biet',         'unknown co y nghia rieng'),
    ('day',       'So roi rac 1-31',               'Bo (nhieu cao, it thong tin cho ARM)',           'Nhieu gia tri, kho tao item co y nghia'),
    ('month',     'Nominal (thang)',               'Giu nguyen',                                    'Da roi rac, co y nghia theo mua'),
    ('duration',  'Lien tuc - DATA LEAKAGE',       '** LOAI KHOI TAP ITEM **',                      'Chi biet sau khi goi xong - khong dung cho ARM'),
    ('campaign',  'So roi rac',                    'Discretize thanh nhom (1 / 2-3 / 4+)',          'Can roi rac hoa de tao item'),
    ('pdays',     'So: -1 = chua lien he truoc',   'Tao bien nhi phan contacted_before (yes/no)',   '-1 khong phai so do binh thuong'),
    ('previous',  'So roi rac',                    'Giu dang 0 / 1+ (co hay khong lien he truoc)',  'Tuong tu pdays'),
    ('poutcome',  'Co gia tri unknown',             'Giu unknown la mot gia tri rieng biet',         'unknown = chua co chien dich truoc'),
    ('y',         'Bien muc tieu binary',           'Giu nguyen, dua vao RHS cua luat ket hop',      'Chi xuat hien o ve phai (consequent)'),
]

plan_df = pd.DataFrame(processing_plan, columns=['Bien', 'Van de', 'Quyet dinh xu ly', 'Ly do'])
plan_df.set_index('Bien', inplace=True)

print('=== BANG QUYET DINH XU LY TUNG BIEN (task 2.3) ===')
print(plan_df.to_string())

# Lưu ra CSV
plan_df.to_csv(PROCESSED_DATA_DIR / 'preprocessing_decision_table.csv', encoding='utf-8-sig')
print(f"\nDa luu: {PROCESSED_DATA_DIR / 'preprocessing_decision_table.csv'}")

=== BANG QUYET DINH XU LY TUNG BIEN (task 2.3) ===
                                Van de                               Quyet dinh xu ly                                           Ly do
Bien                                                                                                                                 
age                           Lien tuc    Discretize thanh nhom (young/middle/senior)          Can de tao item dang age=young cho ARM
job                 Co gia tri unknown          Giu unknown la mot gia tri rieng biet          unknown co y nghia rieng, khong impute
marital      Nominal, khong co unknown             Giu nguyen, encode attribute=value                           Da la gia tri roi rac
education  Co gia tri unknown, ordinal          Giu unknown la mot gia tri rieng biet          unknown co y nghia rieng, khong impute
default                  Binary yes/no             Giu nguyen, encode attribute=value                           Da la gia tri roi rac
balance    